In [ ]:
# Lab type: write
# Course: DS102 — Data Visualisation and Communication
# Lesson: Time Series Visualisation
# Task: Aggregate and plot daily sales as a time series, add rolling averages, highlight events, and compare trends by category and region.

# Lab: Time Series Visualisation

This lab covers the techniques from Lesson 6 — line charts, rolling averages,
area charts, multi-series comparisons, and event annotations — using the retail
sales dataset.

Work through each section in order. Write your answers in the empty markdown cells.

**Outputs are cleared.** Run each cell to generate the plots.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install matplotlib seaborn pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.0)

np.random.seed(42)
n = 500

df = pd.DataFrame({
    "date": pd.date_range("2023-01-01", periods=n, freq="D"),
    "category": np.random.choice(["Electronics", "Clothing", "Books", "Home"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "sales": np.random.lognormal(mean=6.5, sigma=0.8, size=n).round(2),
    "units": np.random.randint(1, 20, n),
    "customer_age": np.random.normal(38, 10, n).clip(18, 70).astype(int),
    "profit_margin": np.random.beta(3, 7, n).round(3),
})

print(df.shape)
df.head()

## Step 1: Aggregate and plot daily sales

The dataset has one row per transaction per day. Aggregate to daily total sales
before plotting — plotting raw rows would stack multiple transactions per day
and give the wrong picture.

In [ ]:
# Aggregate: daily total sales
daily = df.groupby("date")["sales"].sum().reset_index()
daily.columns = ["date", "total_sales"]

print(f"Date range: {daily['date'].min().date()} → {daily['date'].max().date()}")
print(f"Days: {len(daily)}")
daily.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily["date"], daily["total_sales"], linewidth=1, color="#4c72b0")
ax.set_title("Daily total sales — 2023")
ax.set_ylabel("Sales (£)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
plt.show()

> **Question:** How noisy is the daily signal? Can you see any broad trend
> across the year, or is it too noisy to say?
> Can you spot any obvious seasonal pattern just from this chart?

*(Write your answer here.)*

## Step 2: Rolling averages

Add 7-day and 30-day centred rolling averages to the daily data,
then plot all three series together. Use low alpha for the raw daily line.

In [ ]:
daily["rolling_7d"] = daily["total_sales"].rolling(window=7, center=True).mean()
daily["rolling_30d"] = daily["total_sales"].rolling(window=30, center=True).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily["date"], daily["total_sales"],
        color="#4c72b0", alpha=0.25, linewidth=0.8, label="Daily")
ax.plot(daily["date"], daily["rolling_7d"],
        color="#4c72b0", linewidth=1.5, label="7-day avg")
ax.plot(daily["date"], daily["rolling_30d"],
        color="crimson", linewidth=2, label="30-day avg")
ax.set_title("Daily sales with rolling averages")
ax.set_ylabel("Sales (£)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
plt.show()

> **Question:** What trend does the 30-day average reveal that was hidden in the
> daily noise? Why does the rolling average have missing values at the start and
> end of the series (hint: `center=True`)?

*(Write your answer here.)*

## Step 3: Monthly aggregation and area chart

Resample the data to monthly totals using `resample("MS")`, then build
an area chart (line + filled region) to emphasise volume over time.

In [ ]:
monthly = df.set_index("date").resample("MS")["sales"].sum().reset_index()
monthly.columns = ["date", "total_sales"]

print(monthly)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(monthly["date"], monthly["total_sales"],
                alpha=0.2, color="#4c72b0")
ax.plot(monthly["date"], monthly["total_sales"],
        color="#4c72b0", linewidth=2)
ax.set_title("Monthly total sales — 2023")
ax.set_ylabel("Sales (£)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
plt.show()

> **Question:** Which month had the highest total sales? Which had the lowest?
> Does the area chart make it easier to compare months than the daily line chart?

*(Write your answer here.)*

## Step 4: Monthly sales by category

Group by both `date` and `category`, resample to monthly, and plot each
category as a separate line. Use small marker dots to help readers track
individual data points.

In [ ]:
monthly_by_cat = (
    df.set_index("date")
    .groupby("category")
    .resample("MS")["sales"]
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))

for cat, group in monthly_by_cat.groupby("category"):
    ax.plot(group["date"], group["sales"],
            marker="o", markersize=3, linewidth=1.5, label=cat)

ax.set_title("Monthly sales by category")
ax.set_ylabel("Sales (£)")
ax.legend(title="Category", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
plt.show()

> **Question:** Do any categories grow or decline over the year while others stay flat?
> Are the lines so close together that the chart is hard to read?
> If so, what would you do to make it clearer?

*(Write your answer here.)*

## Step 5: Annotate an event

Mark a hypothetical summer sale event on the monthly chart with a vertical
dashed line and a text label. Also highlight Q4 with a shaded span.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly["date"], monthly["total_sales"], color="#4c72b0", linewidth=2)

# Vertical line: summer sale event
event_date = pd.Timestamp("2023-07-01")
ax.axvline(event_date, color="#dd8452", linestyle="--", linewidth=1, alpha=0.8)
ax.text(event_date, ax.get_ylim()[1] * 0.97, "Summer sale",
        color="#dd8452", fontsize=8, ha="left", va="top")

# Shaded span: Q4
ax.axvspan(pd.Timestamp("2023-10-01"), pd.Timestamp("2023-12-31"),
           alpha=0.08, color="gold", label="Q4")

ax.set_title("Monthly sales with event annotation and Q4 highlight")
ax.set_ylabel("Sales (£)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
plt.show()

> **Question:** Does the Q4 highlight reveal anything different from the rest of the year?
> How does having the annotation change the way you read the chart compared to having
> no context about when events occurred?

*(Write your answer here.)*

## Challenge: weekly aggregation by region

Aggregate the dataset to **weekly** total sales (`resample("W")`) for each region.
Plot each region as a separate line on the same axis.

Then add a 4-week rolling average on top of one region's line to smooth it out.
Write a title that states a finding from the chart.

In [ ]:
# Aggregate: weekly total sales by region
weekly_by_region = (
    df.set_index("date")
    .groupby("region")
    .resample("W")["sales"]
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))

for region, group in weekly_by_region.groupby("region"):
    group = group.set_index("date")
    ax.plot(group.index, group["sales"], linewidth=1.2, alpha=0.7, label=region)

# Add a 4-week rolling average for one region (e.g. North)
north = weekly_by_region[weekly_by_region["region"] == "North"].set_index("date")
north["rolling_4w"] = north["sales"].rolling(window=4, center=True).mean()
ax.plot(north.index, north["rolling_4w"],
        color="black", linewidth=2.5, linestyle="--", label="North 4-week avg")

ax.set_title("Weekly sales by region")
ax.set_ylabel("Sales (£)")
ax.legend(title="Region", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
plt.show()

> **Question:** Does the weekly resolution reveal any pattern that the monthly view obscured?
> Does the rolling average for the North region reveal a trend that wasn't obvious
> from the raw weekly line?

*(Write your answer here.)*